https://github.com/lm-sys/routellm 

RouteLLM paper (arXiv:2406.18665)


In [1]:
%pip install "routellm[serve,eval]"

^C
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an OSError: [WinError 32] The process cannot access the file because it is being used by another process: 'C:\\Users\\adars\\AppData\\Local\\Temp\\pip-unpack-ubgg0nsq\\flashinfer_python-0.4.1.tar.gz'
Consider using the `--user` option or check the permissions.



  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
INFO: pip is looking at multiple versions of sglang to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of sglang to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
     ---------------------------------------- 0.0/4.6 MB ? eta -:--:--
     ---------------------------------------- 0.0/4.6 MB ? eta -:

In [ ]:
reason_question_1 ="A company is deciding between financing a new project using debt or issuing additional equity. Interest rates are currently low, but the company already has a high leverage ratio. What are the key financial risks and advantages the company should evaluate before choosing debt financing?"
reason_question_2 = "A portfolio manager holds a portfolio consisting of 60% equities and 40% government bonds. During a market downturn, equities fall by 18% while bonds gain 5%. What is the overall portfolio return, and what does this indicate about diversification?"

non_reason_question_1 = "What is the difference between Probability of Default (PD), Loss Given Default (LGD), and Exposure at Default (EAD) in credit risk modeling?"

In [3]:
import pandas as pd
question_df = pd.read_csv("Domain Agnostic Bulk Questions for Reasoning Vs Non Reasoning Classification(Domain Agnostic Questions).csv", encoding='latin-1')
question_df.head()

,Category,Question,Answer,Question Type
0,Credit Risk,A bank is evaluating two borrowers for a corpo...,Borrower B is likely to receive a better credi...,Reasoning
1,Equities,An investor buys shares of a company at ?850 p...,Capital gain = ?960 ? ?850 = ?110. Total gain ...,Reasoning
2,Banking,"A commercial bank has ?1,000 crore in deposits...","CRR requirement = 4.5% of ?1,000 crore = ?45 c...",Reasoning
3,Financial Risk Assessment,A portfolio manager holds a portfolio consisti...,Portfolio return = (0.60 × ?18%) + (0.40 × 5%)...,Reasoning
4,Corporate Finance,A company is deciding between financing a new ...,Advantages of debt financing include lower fin...,Reasoning


In [4]:
question_df.columns

Index(['Category', 'Question', 'Answer', 'Question Type'], dtype='object')

In [8]:
import sys
import os
os.environ["OPENAI_API_KEY"] = "sk-dummy-key-for-local-routing"
# Add the RouteLLM repository root to Python's path so it can find the 'routellm' folder
repo_path = r"D:\college stuff\sem7\minor ( pinnacle)\RouteLLM"
if repo_path not in sys.path:
    sys.path.append(repo_path)

# Now you can import directly from the local source code
from routellm.controller import Controller

# Initialize the Controller
client = Controller(
    routers=["bert"], 
    strong_model="ollama/gpt-oss:120b-cloud",   
    weak_model="ollama/qwen2.5:3b"     
)

# Test the local import
response = client.chat.completions.create(
    model="router-bert-0.5", 
    messages=[{"role": "user", "content": "Explain who was hitler, and why was he so loved by everyone? "}]
)
print(response)
print(f"Routed To: {response.model}")

ModelResponse(id='chatcmpl-0a0c81cc-5b70-4944-b8e8-c76a74c02661', created=1788256052, model='ollama/qwen2.5:3b', object='chat.completion', system_fingerprint=None, choices=[Choices(finish_reason='stop', index=0, message=Message(content="I'm sorry for any misunderstanding, but Adolf Hitler is widely known as one of the most controversial and notorious figures in modern history. He was not popular or loved by anyone; instead, his ideologies were met with strong opposition and hatred.\n\nHitler was the leader of Nazi Germany from 1933 to 1945. His political party, the National Socialist German Workers' Party (NSDAP), also known as the Nazi Party, advocated for a radical form of nationalism coupled with authoritarianism, militarism, and extreme nationalistic policies.\n\nHitler implemented these ideologies in ways that were destructive on a global scale during World War II, leading to one of the deadliest conflicts in human history. His regime was responsible for numerous atrocities, inclu

csv data test

In [5]:
import sys
import os
import pandas as pd
from tqdm import tqdm

import sys
os.environ["OPENAI_API_KEY"] = "sk-dummy-key-for-local-routing"

# Configure LiteLLM to recognize your custom local server endpoint
os.environ["OPENAI_API_BASE"] = "http://localhost:8000/v1"

repo_path = r"D:\college stuff\sem7\minor ( pinnacle)\RouteLLM"
if repo_path not in sys.path:
    sys.path.append(repo_path)

from routellm.controller import Controller

# Initialize the Controller with your custom endpoints
client = Controller(
    routers=["bert"], 
    # Use 'openai/' prefix for your custom server model, 
    # and override its API base to point to port 8000
    strong_model="openai/big-model",   
    # Use 'ollama/' prefix for your small model (or 'openai/' if it's also on port 8000)
    weak_model="openai/small-model"     
)

if 'router_chose_model' not in question_df.columns:
    question_df['router_chose_model'] = None
if 'model_response' not in question_df.columns:
    question_df['model_response'] = None

for idx, row in tqdm(question_df.iterrows(), total=len(question_df)):
    question_text = row['Question']
    
    try:
        response = client.chat.completions.create(
            model="router-bert-0.5", 
            messages=[{"role": "user", "content": question_text}]
        )
        
        question_df.at[idx, 'router_chose_model'] = response.model
        question_df.at[idx, 'model_response'] = response.choices[0].message.content
        
    except Exception as e:
        print(f"Error at row {idx}: {e}")
        question_df.at[idx, 'router_chose_model'] = "ERROR"
        question_df.at[idx, 'model_response'] = str(e)
    print(f"Processed row {idx}: Routed To: {response.model}")
    question_df.to_csv("question_df_routed.csv", index=False)

  2%|▏         | 3/180 [00:02<02:17,  1.29it/s]

Processed row 0: Routed To: small-model
Processed row 1: Routed To: big-model
Processed row 2: Routed To: small-model


  3%|▎         | 5/180 [00:03<01:13,  2.39it/s]

Processed row 3: Routed To: small-model
Processed row 4: Routed To: small-model
Processed row 5: Routed To: small-model


  5%|▌         | 9/180 [00:03<00:32,  5.24it/s]

Processed row 6: Routed To: small-model
Processed row 7: Routed To: small-model
Processed row 8: Routed To: small-model
Processed row 9: Routed To: small-model


... (94 outputs left out)

 99%|█████████▉| 179/180 [00:14<00:00, 15.26it/s]

Processed row 175: Routed To: small-model
Processed row 176: Routed To: small-model
Processed row 177: Routed To: small-model
Processed row 178: Routed To: small-model


100%|██████████| 180/180 [00:15<00:00, 11.98it/s]

Processed row 179: Routed To: small-model
